# 🗓️ Conflict Resolver — SFT + GRPO Training
**Runtime: T4 GPU** | Full pipeline: Install → Baseline → SFT → GRPO → Compare

In [ ]:
# Cell 1: Install everything
!pip install -q unsloth trl datasets pydantic fastapi huggingface_hub nest_asyncio
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Cell 2: Download environment + setup
from huggingface_hub import snapshot_download
import sys, os, asyncio, json, random
import nest_asyncio
nest_asyncio.apply()

env_path = snapshot_download(
    repo_id="srivtx/openenv-conflict-resolver-v2",
    repo_type="space", local_dir="./env_code"
)
sys.path.insert(0, os.path.join(env_path, "src"))

from assistant_conflict_env.environment import PersonalAssistantConflictEnv
from assistant_conflict_env.models import ConflictAction, ActionIntent, Owner, Priority
from assistant_conflict_env.eval_set import (
    train_episodes, holdout_episodes, adversarial_episodes,
)

# Disjoint procedural pools. Train and holdout share NO seeds by construction
# (see assistant_conflict_env/eval_set.py). Honest generalization == holdout.
TRAIN_TASK_IDS = [t.id for t in train_episodes(difficulty="hard", limit=80)]
HOLDOUT_TASK_IDS = [t.id for t in holdout_episodes(difficulty="hard", limit=40)]
ADVERSARIAL_TASK_IDS = [t.id for t in adversarial_episodes(difficulty="hard")]

# Legacy static demo tasks kept only for the deployed UI; do not train or eval
# the model on these because the static answer key is too small.
LEGACY_DEMO_TASK_IDS = [
    "easy_evening_planner", "medium_multi_party_negotiation", "hard_cascade_replanning",
]

print(f"Train pool:       {len(TRAIN_TASK_IDS):>3} procedural episodes (seeds 1000-1999)")
print(f"Holdout pool:     {len(HOLDOUT_TASK_IDS):>3} procedural episodes (seeds 9000-9099)")
print(f"Adversarial pool: {len(ADVERSARIAL_TASK_IDS):>3} procedural episodes (seeds 5000-5009)")

# Quick smoke test on the first train task.
env = PersonalAssistantConflictEnv()
r = await env.reset(task_name=TRAIN_TASK_IDS[0])
print(f"Environment loaded! First conflict: {r.observation.current_conflict.summary[:90]}...")

In [ ]:
# Cell 3: Helper functions

def heuristic_action(obs):
    c = obs.current_conflict
    if c is None:
        return ConflictAction(intent="route_message", owner="self", priority="normal", message_template="Default.")
    text = f"{c.summary} {' '.join(c.constraints)}".lower()
    if any(t in text for t in ["missing", "unclear", "timezone", "attachment"]):
        return ConflictAction(intent="ask_clarification", owner="work", priority="high", needs_clarification=True, message_template="Need clarification before execution.")
    elif any(t in text for t in ["overlap", "delay", "reservation", "check-in", "driver"]):
        return ConflictAction(intent="reschedule_event", owner="travel", priority="high", proposed_slot="after 20:30", message_template="Reschedule to protect constraints.")
    elif any(t in text for t in ["pickup", "gift", "cancel window"]):
        return ConflictAction(intent="delegate_task", owner="family", priority="normal", message_template="Delegate with confirmation.")
    elif any(t in text for t in ["payment", "renewal", "insurance"]):
        return ConflictAction(intent="route_message", owner="finance", priority="urgent", message_template="Route payment urgently.")
    elif any(t in text for t in ["final", "consolidated", "itinerary"]):
        return ConflictAction(intent="finalize_itinerary", owner="self", priority="high", message_template="Finalize timeline with risks.")
    return ConflictAction(intent="route_message", owner="self", priority="normal", message_template="Routing with fallback.")

def parse_json(text):
    text = (text or "").strip()
    try: return json.loads(text)
    except: pass
    l, r = text.find("{"), text.rfind("}")
    if l >= 0 and r > l:
        try: return json.loads(text[l:r+1])
        except: pass
    return None

async def eval_model(mdl, tok, label, task_ids=None):
    """Evaluate the model on an explicit list of task ids.

    Always pass a task_ids list; defaulting to the legacy demo tasks would
    re-introduce train/eval overlap. Use HOLDOUT_TASK_IDS for honest numbers.
    """
    if task_ids is None:
        raise ValueError("eval_model requires task_ids; use HOLDOUT_TASK_IDS for honest numbers.")
    env = PersonalAssistantConflictEnv()
    results = {}
    for tid in task_ids:
        r = await env.reset(task_name=tid)
        while not r.done and r.observation.current_conflict:
            c = r.observation.current_conflict
            prompt = (
                "Return ONLY a JSON object, nothing else.\n"
                '{"intent": "...", "owner": "...", "priority": "...", '
                '"proposed_slot": "...", "needs_clarification": false, '
                '"message_template": "..."}\n\n'
                f"Conflict: {c.summary}\n"
                f"Constraints: {', '.join(c.constraints)}\n"
                "Intents: route_message/propose_plan/reschedule_event/delegate_task/ask_clarification/finalize_itinerary\n"
                "Owners: self/work/family/travel/finance/legal\n"
                "Priorities: low/normal/high/urgent\nJSON:"
            )
            msgs = [{"role": "user", "content": prompt}]
            inp = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
            out = mdl.generate(inp, max_new_tokens=200, temperature=0.05, do_sample=True, repetition_penalty=1.1)
            text = tok.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
            p = parse_json(text)
            if p:
                try:
                    action = ConflictAction(
                        intent=str(p.get("intent","route_message")).lower(),
                        owner=str(p.get("owner","self")).lower(),
                        priority=str(p.get("priority","normal")).lower(),
                        proposed_slot=str(p.get("proposed_slot",""))[:80],
                        needs_clarification=bool(p.get("needs_clarification",False)),
                        message_template=str(p.get("message_template","Act."))[:500]
                    )
                except: action = heuristic_action(r.observation)
            else: action = heuristic_action(r.observation)
            r = await env.step(action)
        s = await env.state()
        score = float(s.final_score or s.average_step_score)
        results[tid] = round(score, 4)
        print(f"  [{label}] {tid}: {score:.4f}")
    return results

print("Helpers ready!")

In [ ]:
# Cell 4: Load fresh Qwen 3B
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=608, load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0, use_gradient_checkpointing="unsloth"
)
print("Model loaded!")

In [ ]:
# Cell 5: Evaluate UNTRAINED baseline on the HELD-OUT pool.
# This is the honest "before" number: the untrained model has never seen
# these episodes (and never will during training).
FastLanguageModel.for_inference(model)
print("Evaluating untrained 3B on holdout pool...")
pretrain_scores = await eval_model(model, tokenizer, "Untrained", task_ids=HOLDOUT_TASK_IDS)
avg_pre = sum(pretrain_scores.values()) / max(len(pretrain_scores), 1)
print(f"\nUntrained holdout average: {avg_pre:.4f} (n={len(pretrain_scores)})")

In [ ]:
# Cell 6: Build SFT dataset from PROCEDURAL train episodes.
#
# Key differences from the previous version (which Kimi correctly criticized):
#   * Source is the train pool (80 unique procedural episodes) instead of the
#     3 static fixtures, so SFT teaches the *format* and *intent routing*, not
#     the answers to specific eval cases.
#   * No `* 15` duplication. Each example appears exactly once. Diversity of
#     conflicts is what gives a real training signal.
#   * Answer text is varied per-example; we do NOT inject the required_keywords
#     verbatim (that was the keyword-stuffing shortcut).
#   * On clarification cases we generate TWO SFT examples: one for the initial
#     ask_clarification, one for the post-reveal action. This teaches the model
#     to operate in the two-step partial-observability regime.
sft_data = []
for tid in TRAIN_TASK_IDS:
    env = PersonalAssistantConflictEnv()
    r = await env.reset(task_name=tid)
    while not r.done and r.observation.current_conflict:
        c = r.observation.current_conflict
        # Resolve the expected expectation, switching to post-reveal when the
        # env has already revealed clarification info for this conflict.
        revealed = bool(c.revealed_info)
        exp = c.expected
        if revealed and c.clarification and c.clarification.post_reveal_expected:
            exp = c.clarification.post_reveal_expected

        prompt = (
            "Return ONLY a JSON object, nothing else.\n"
            '{"intent": "...", "owner": "...", "priority": "...", '
            '"proposed_slot": "...", "needs_clarification": false, '
            '"message_template": "..."}\n\n'
            f"Conflict: {c.summary}\n"
            f"Constraints: {', '.join(c.constraints)}\n"
            "Intents: route_message/propose_plan/reschedule_event/delegate_task/ask_clarification/finalize_itinerary\n"
            "Owners: self/work/family/travel/finance/legal\n"
            "Priorities: low/normal/high/urgent\nJSON:"
        )
        intent_val = exp.intent.value if hasattr(exp.intent, 'value') else str(exp.intent)
        owner_val = exp.owner.value if hasattr(exp.owner, 'value') else str(exp.owner)
        priority_val = exp.priority.value if hasattr(exp.priority, 'value') else str(exp.priority)
        # Generate a slightly varied, on-topic message instead of dumping
        # required_keywords verbatim (that was the gameable shortcut).
        if intent_val == "ask_clarification":
            msg = f"Need to clarify the missing detail before scheduling; will follow up with the owner."
        elif intent_val == "reschedule_event":
            msg = f"Reschedule to {exp.expected_slot_hint or 'a confirmed slot'} and notify the owner with the new context."
        elif intent_val == "delegate_task":
            msg = f"Delegate to the {owner_val} owner with confirmation; flag the constraint and follow up tonight."
        elif intent_val == "route_message":
            msg = f"Route to the {owner_val} owner with priority {priority_val}; include the deadline and constraint."
        elif intent_val == "propose_plan":
            msg = f"Propose plan with target slot {exp.expected_slot_hint or 'tbd'}; include risk note and fallback path."
        else:  # finalize_itinerary
            msg = f"Finalize itinerary with timeline, owners, risk notes, and fallback path documented."
        answer = json.dumps({
            "intent": intent_val,
            "owner": owner_val,
            "priority": priority_val,
            "proposed_slot": exp.expected_slot_hint or "",
            "needs_clarification": (exp.block_if_missing_context and not revealed),
            "message_template": msg,
        })
        text = tokenizer.apply_chat_template([
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": answer}
        ], tokenize=False)
        sft_data.append({"text": text})
        # Step the env using the oracle expectation so the conflict queue
        # advances naturally (and clarification reveals fire when due).
        oracle_action = ConflictAction(
            intent=intent_val, owner=owner_val, priority=priority_val,
            proposed_slot=exp.expected_slot_hint or "",
            needs_clarification=(exp.block_if_missing_context and not revealed),
            message_template=msg,
        )
        r = await env.step(oracle_action)

random.Random(42).shuffle(sft_data)
print(f"SFT dataset: {len(sft_data)} unique examples (NO duplication, source=procedural train pool).")
print(f"\nExample answer:\n{sft_data[0]['text'][-300:]}")

In [ ]:
# Cell 7 (intentionally empty after rebuild).
# The previous notebook had a duplicate fragment of Cell 6 here that did
# nothing because Cell 6 had already executed. The rebuilt Cell 6 below
# generates SFT data procedurally in a single pass, so this cell is now a
# no-op kept only to preserve cell numbering in older training logs.
print("Cell 7: no-op (formerly duplicate fragment of Cell 6).")

In [ ]:
# Cell 7: SFT Training — teach correct JSON format (~5-10 min)
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

sft_dataset = Dataset.from_list(sft_data)

sft_trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    args=SFTConfig(
        output_dir="./sft_output",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        max_seq_length=608,
        logging_steps=10,
        warmup_steps=5,
        seed=42,
    ),
    processing_class=tokenizer,
)

print("SFT training (teaching correct answers)...")
sft_trainer.train()
print("SFT complete!")

In [ ]:
# Cell 8: Evaluate AFTER SFT on the HELD-OUT pool.
FastLanguageModel.for_inference(model)
print("Evaluating SFT-trained model on holdout pool...")
sft_scores = await eval_model(model, tokenizer, "SFT", task_ids=HOLDOUT_TASK_IDS)
avg_sft = sum(sft_scores.values()) / max(len(sft_scores), 1)
print(f"\nSFT holdout average: {avg_sft:.4f} (was {avg_pre:.4f}; n={len(sft_scores)})")

In [ ]:
# Cell 9: GRPO Training on top of SFT (~30-40 min)
from trl import GRPOConfig, GRPOTrainer

# Collect prompts for GRPO
async def collect_prompts(episodes=60):
    """Collect (prompt, task_name, history) rows from procedural TRAIN episodes.

    The reward function below replays the env up to each step, applies the
    sampled completion as an action, and uses the env's REAL reward as the
    GRPO reward signal. No placeholder rewards.
    """
    env = PersonalAssistantConflictEnv()
    rows = []
    train_ids = TRAIN_TASK_IDS  # disjoint from HOLDOUT and ADVERSARIAL pools
    for ep in range(episodes):
        tid = train_ids[ep % len(train_ids)]
        r = await env.reset(task_name=tid)
        hist = []
        while not r.done and r.observation.current_conflict:
            c = r.observation.current_conflict
            prompt = (
                "Return ONLY a JSON object, nothing else.\n"
                '{"intent": "...", "owner": "...", "priority": "...", '
                '"proposed_slot": "...", "needs_clarification": false, '
                '"message_template": "..."}\n\n'
                f"Conflict: {c.summary}\n"
                f"Constraints: {', '.join(c.constraints)}\n"
                "Intents: route_message/propose_plan/reschedule_event/delegate_task/ask_clarification/finalize_itinerary\n"
                "Owners: self/work/family/travel/finance/legal\n"
                "Priorities: low/normal/high/urgent\nJSON:"
            )
            rows.append({"prompt": prompt, "task_name": tid, "history": [dict(h) for h in hist], "step": r.observation.step_index})
            action = heuristic_action(r.observation)
            r = await env.step(action)
            hist.append(action.model_dump(mode="json"))
            if len(rows) >= 180: break
        if len(rows) >= 180: break
    return rows

prompt_rows = await collect_prompts()

def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for i, comp in enumerate(completions):
        text = comp[0]["content"] if isinstance(comp, list) else str(comp)
        p = parse_json(text)
        if p is None:
            rewards.append(0.0); continue
        try:
            action = ConflictAction(
                intent=str(p.get("intent","route_message")).lower(),
                owner=str(p.get("owner","self")).lower(),
                priority=str(p.get("priority","normal")).lower(),
                proposed_slot=str(p.get("proposed_slot",""))[:80],
                needs_clarification=bool(p.get("needs_clarification",False)),
                message_template=str(p.get("message_template","Act."))[:500]
            )
            row = prompt_rows[min(i, len(prompt_rows)-1)]
            env_l = PersonalAssistantConflictEnv()
            res = asyncio.run(env_l.reset(task_name=row["task_name"]))
            for h in row.get("history", []):
                if res.done: break
                res = asyncio.run(env_l.step(ConflictAction.model_validate(h)))
            if not res.done:
                res = asyncio.run(env_l.step(action))
                rewards.append(float(res.reward))
            else: rewards.append(0.0)
        except: rewards.append(0.0)
    return rewards

grpo_dataset = Dataset.from_list(prompt_rows)

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=GRPOConfig(
        output_dir="./grpo_out",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,
        num_train_epochs=1,
        logging_steps=5,
        max_prompt_length=512,
        max_completion_length=96,
        seed=42,
    ),
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
)

print("GRPO training (1 epoch, conservative LR)...")
grpo_trainer.train()
print("GRPO complete!")

In [ ]:
# Cell 10: Save final model
model.save_pretrained("./trained_conflict_resolver")
tokenizer.save_pretrained("./trained_conflict_resolver")
print("Model saved!")

In [ ]:
# Cell 11: Evaluate FINAL trained model on the HELD-OUT pool and record
# adversarial-pool numbers for the shortcut-shutdown probe.
FastLanguageModel.for_inference(model)
print("Evaluating final model (SFT + GRPO) on holdout pool...")
final_scores = await eval_model(model, tokenizer, "Final", task_ids=HOLDOUT_TASK_IDS)
print("\nEvaluating final model on adversarial pool...")
adversarial_scores = await eval_model(model, tokenizer, "Adversarial", task_ids=ADVERSARIAL_TASK_IDS)

In [ ]:
# Cell 12: FINAL COMPARISON TABLES
#
# All numbers below are on PROCEDURAL HOLDOUT episodes (seeds disjoint from
# training). This is the honest generalization metric.

def _avg(d):
    vals = list(d.values())
    return sum(vals) / max(len(vals), 1)

print("\n" + "=" * 78)
print(f"HOLDOUT POOL ({len(HOLDOUT_TASK_IDS)} episodes, seeds 9000-9099)")
print("=" * 78)
print(f"{'Task':<32} {'Before':>11} {'SFT':>11} {'SFT+GRPO':>11}")
print("-" * 78)
# Show the first 8 episodes for compactness; the average is over all.
sample_ids = HOLDOUT_TASK_IDS[:8]
for t in sample_ids:
    b = pretrain_scores.get(t, float("nan"))
    s = sft_scores.get(t, float("nan"))
    f = final_scores.get(t, float("nan"))
    print(f"{t:<32} {b:>11.4f} {s:>11.4f} {f:>11.4f}")
print("-" * 78)
print(f"{'AVERAGE (n=' + str(len(HOLDOUT_TASK_IDS)) + ')':<32} "
      f"{_avg(pretrain_scores):>11.4f} {_avg(sft_scores):>11.4f} {_avg(final_scores):>11.4f}")
print(f"{'Improvement over baseline':<32} {'':>11} "
      f"{_avg(sft_scores)-_avg(pretrain_scores):>+10.4f} {_avg(final_scores)-_avg(pretrain_scores):>+10.4f}")

print("\n" + "=" * 78)
print(f"ADVERSARIAL POOL ({len(ADVERSARIAL_TASK_IDS)} episodes, seeds 5000-5009)")
print("Verifies the substring-slot and keyword-stuffing shortcuts are dead.")
print("=" * 78)
print(f"{'AVERAGE (final model on adversarial pool)':<48} {_avg(adversarial_scores):>11.4f}")
if _avg(adversarial_scores) >= _avg(final_scores) + 0.05:
    print("WARNING: adversarial score noticeably > holdout score; double-check graders.")

print("\nTraining pipeline: Untrained -> SFT (procedural data, format + routing)")
print("                    -> GRPO (env reward, 1 conservative epoch)")
print("All evaluation numbers above are on UNSEEN procedural episodes.")

In [ ]:
# Cell 13 (added in rebuild): adversarial grader probes.
#
# These don't train anything; they verify directly that the rebuilt grader has
# closed the substring-slot shortcut and the keyword-stuffing shortcut. If
# either shortcut were still open, we'd see suspiciously high scores in the
# adversarial pool above.
from assistant_conflict_env.graders import _slot_score, _message_score
from assistant_conflict_env.models import ActionIntent

shortcut_examples = [
    ("after 20:30", "later today",            "OLD shortcut: 'today' substring"),
    ("after 20:30", "3pm tomorrow",           "OLD shortcut: 'pm' substring"),
    ("20:30",        "reschedule to 20:30",   "Honest match"),
    ("20:30",        "21:00",                  "30 min off"),
    ("20:30",        "08:00",                  "Wildly wrong"),
]
print("Slot-score probe (shortcuts should be 0.0; honest matches should be 1.0):")
for hint, slot, label in shortcut_examples:
    s = _slot_score(hint, slot, require_slot=True)
    print(f"  hint={hint!r:<20} slot={slot!r:<28} -> {s:.2f}  ({label})")

print("\nMessage-score probe (stuffing should be lower than a real message):")
stuffed   = "reschedule, work, urgent."
real_msg  = "Reschedule the incident review to 20:30 with owner confirmation and follow up note."
for label, text in [("STUFFED", stuffed), ("REAL", real_msg)]:
    score = _message_score(ActionIntent.RESCHEDULE_EVENT, text)
    print(f"  {label:<8}: {score:.2f}  ({text!r})")

In [ ]:
# Cell 14: SFT loss curve (training evidence #1).
#
# Pulls the per-step loss from `sft_trainer.state.log_history` and renders
# a line chart, then saves the PNG to ./assets/sft_loss.png so judges have
# a static artifact even outside the notebook.
import os
import matplotlib.pyplot as plt

os.makedirs("./assets", exist_ok=True)

sft_log = getattr(sft_trainer.state, "log_history", []) or []
sft_steps  = [r["step"] for r in sft_log if "loss" in r]
sft_losses = [r["loss"] for r in sft_log if "loss" in r]

fig, ax = plt.subplots(figsize=(7, 4))
if sft_losses:
    ax.plot(sft_steps, sft_losses, linewidth=2, color="#4C9AFF")
    ax.set_title(f"SFT training loss  (n={len(sft_losses)} log points)")
else:
    ax.text(0.5, 0.5, "No SFT loss entries in log_history", ha="center", va="center")
    ax.set_title("SFT training loss (no data)")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig("./assets/sft_loss.png", dpi=150)
plt.show()
print(f"Saved ./assets/sft_loss.png  (points={len(sft_losses)})")

In [ ]:
# Cell 15: GRPO reward curve (training evidence #2).
#
# Pulls per-step reward stats from `grpo_trainer.state.log_history`. TRL's
# GRPOTrainer logs both `reward` (mean over the group) and, on newer
# versions, `reward_std`. We plot whichever is present so this works across
# TRL releases.
grpo_log = getattr(grpo_trainer.state, "log_history", []) or []

def _extract(key):
    return (
        [r["step"] for r in grpo_log if key in r],
        [r[key]  for r in grpo_log if key in r],
    )

reward_steps, reward_vals = _extract("reward")
loss_steps,   loss_vals   = _extract("loss")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

if reward_vals:
    axes[0].plot(reward_steps, reward_vals, linewidth=2, color="#36B37E")
    axes[0].set_title(f"GRPO mean reward per logged step  (n={len(reward_vals)})")
else:
    axes[0].text(0.5, 0.5, "No 'reward' entries in log_history", ha="center", va="center")
    axes[0].set_title("GRPO reward (no data)")
axes[0].set_xlabel("step"); axes[0].set_ylabel("reward")
axes[0].grid(alpha=0.25)

if loss_vals:
    axes[1].plot(loss_steps, loss_vals, linewidth=2, color="#FF8B00")
    axes[1].set_title(f"GRPO policy loss  (n={len(loss_vals)})")
else:
    axes[1].text(0.5, 0.5, "No 'loss' entries in log_history", ha="center", va="center")
    axes[1].set_title("GRPO loss (no data)")
axes[1].set_xlabel("step"); axes[1].set_ylabel("loss")
axes[1].grid(alpha=0.25)

fig.tight_layout()
fig.savefig("./assets/grpo_reward_loss.png", dpi=150)
plt.show()
print(f"Saved ./assets/grpo_reward_loss.png  (reward_pts={len(reward_vals)}, loss_pts={len(loss_vals)})")

In [ ]:
# Cell 16: Holdout + adversarial score comparison (training evidence #3).
#
# Bar chart of the same numbers printed by Cell 12 — visual evidence of
# untrained baseline -> SFT -> SFT+GRPO -> adversarial. Saved to
# ./assets/score_comparison.png.
import numpy as np

def _safe_avg(d):
    vals = list(d.values()) if isinstance(d, dict) else []
    return sum(vals) / max(len(vals), 1)

scores = {
    "Untrained (holdout)":  _safe_avg(pretrain_scores),
    "SFT (holdout)":         _safe_avg(sft_scores),
    "SFT+GRPO (holdout)":    _safe_avg(final_scores),
    "SFT+GRPO (adversarial)": _safe_avg(adversarial_scores),
}

labels = list(scores.keys())
values = list(scores.values())
colors = ["#FF5630", "#FFAB00", "#36B37E", "#0052CC"]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(labels, values, color=colors, width=0.6)
ax.set_ylim(0.0, 1.05)
ax.set_ylabel("avg env reward (0-1)")
ax.set_title("Holdout + adversarial scores by training stage")
ax.grid(axis="y", alpha=0.25)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10)

plt.setp(ax.get_xticklabels(), rotation=12, ha="right")
fig.tight_layout()
fig.savefig("./assets/score_comparison.png", dpi=150)
plt.show()

print("\nFinal numbers:")
for k, v in scores.items():
    print(f"  {k:<26} {v:.4f}")
print("\nSaved ./assets/score_comparison.png")